In [1]:
import numpy as np 
import pandas as pd 
import re
from nltk.corpus import stopwords
import nltk
from nltk.stem import PorterStemmer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ammar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
data_path = r"C:\\Users\\Ammar\\OneDrive\\Dokumen\\Kuylah\\SEM 2\\RFPP\\RFPP_Task\\Task N-grams\\abstract.csv"
df = pd.read_csv(data_path, sep=";")
df = df.drop(columns=['abstrak_idn'])
df.head()

,title,abstrak_eng
0,IndoBERT Optimization for Sentiment Analysis o...,"In the digital era, sentiment analysis ha..."
1,Offensive Language and Hate Speech Detection U...,Hate speech detection is a crucial issue...
2,Developments and Trends in Indonesian Tourism ...,"Information technology has changed society, se..."
3,Personality Classification of Myers Briggs Typ...,Personality classification using textual data ...


In [3]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, stop_words=None):
        self.stop_words = stop_words if stop_words else set()
        self.stemmer = PorterStemmer()

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        cleaned_texts = []
        for text in X:
            text = text.lower()
            text = re.sub(r'\d+', '', text)
            text = re.sub(r'[^\w\s]', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
            tokens = text.split()
            tokens = [word for word in tokens if word not in self.stop_words]
            tokens = [self.stemmer.stem(word) for word in tokens]
            # cleaned_texts.append(tokens)
            cleaned_texts.append(" ".join(tokens))
        return cleaned_texts

class TrigramExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        trigrams_docs = []
        for tokens in X:
            trigrams = [
                " ".join(tokens[i:i+3])
                for i in range(len(tokens)-2)
            ]
            trigrams_docs.append(trigrams)
        return trigrams_docs

stop_words = set(stopwords.words('english'))

In [4]:
pipeline = Pipeline([
    ("cleaning", TextPreprocessor(stop_words=stop_words)),
    # ("trigram", TrigramExtractor())
])

In [5]:
preprocessed_output = pipeline.fit_transform(df['abstrak_eng'])
print(preprocessed_output)

['digit era sentiment analysi becom crucial evalu public opinion especi context play storeapp indonesianlanguag review research aim improv perform indobert model sentiment analysi deepseek app review use data augment hyperparamet tune techniqu data augment done backtransl techniqu hyperparamet test includ number epoch learn rate batch size experiment result show combin data augment epoch learn rate e batch size produc thehighest accuraci fscore better stabil model without augment model without augment show fluctuat perform indic overfit configur find confirm import appli augment techniqu hyperparamet tune improv accuraci stabil sentiment analysi model contribut develop nlp model indonesian resourceconstrain languag', 'hate speech detect crucial issu sentiment analysi natur languag process studi aim improv effect hate speech detect english text util bert model addit modifi preprocess techniqu develop enhanc fscore dataset use studi sourc kaggl contain english text hate speech content ev

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(3, 3)) # Pembuatan 3-gram
tfidf_matrix = vectorizer.fit_transform(preprocessed_output)
print(tfidf_matrix.shape)

(4, 445)


In [7]:
query = "Personality classification using textual data"
stemmed_q = [PorterStemmer().stem(word) for word in query.split()]
stemmed_q = " ".join(stemmed_q)
query_vector = vectorizer.transform([stemmed_q])

scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

In [8]:
print(scores)

[0.         0.         0.         0.20687248]
